# Actual NeMo retrieval and action boundaries

Use this folder/shared kernel. All policy/proposal data is fictional synthetic fixture data. Canonical intents are explicitly supplied, with no language-model classification. No API keys or model services are used.

In [ ]:
import os
os.environ.update(DO_NOT_TRACK="1", OTEL_SDK_DISABLED="true", HF_HUB_OFFLINE="1", LITELLM_LOCAL_MODEL_COST_MAP="true")
import json
from pathlib import Path
from pprint import pprint
fixtures = json.loads(Path("docs/dataset/actions.json").read_text())
retrieved_text = Path("docs/dataset/policy.txt").read_text()
pprint(fixtures)

## Read the Colang flows and registered actions

`tools/native_flows.py` registers ordinary async actions with actual LLMRails. `config/rails.co` enables retrieval masking and orders validation → application confirmation → local mutation. Context identity/permission/confirmation comes from app-owned fields, never proposal claims. Read the helper and configuration before importing.

In [ ]:
from tools.native_flows import build_rails, retrieve_fixture, submit_fixture, bot_messages, action_outcome
print(Path("config/rails.co").read_text())

## Real retrieval rail, before and after

The helper sends ContextUpdate plus BotIntent through generate_events_async. Inspect actual mask_context action-finished and ContextUpdate evidence and final BotMessage. This is a supplied tiny context, not an indexed search or model answer.

In [ ]:
retrieval_log, retrieval_actions = [], []
retrieval_rails = build_rails(fixtures[0]["proposal"], fixtures[0]["application_context"], retrieval_log, retrieval_actions)
retrieval_events = await retrieve_fixture(retrieval_rails, retrieved_text)
pprint({"before": retrieved_text, "registered_actions": retrieval_actions, "final_bot_messages": bot_messages(retrieval_events)})
pprint([event for event in retrieval_events if event["type"] == "ContextUpdate" or (event["type"] == "InternalSystemActionFinished" and event["action_name"] == "mask_context")])
assert "@" not in bot_messages(retrieval_events)[0]
assert retrieval_log == []

## Real action flow with required confirmation

Every fixture supplies the same canonical submit-leave intent. The framework invokes registered validation actions. Only authorized, valid, app-confirmed proposals should mutate the local log.

In [ ]:
baseline = {}
for item in fixtures:
    action_log, action_events = [], []
    rails = build_rails(item["proposal"], item["application_context"], action_log, action_events, require_confirmation=True)
    native_events = await submit_fixture(rails)
    baseline[item["id"]] = action_outcome(action_log, action_events, native_events)
for name, result in baseline.items():
    pprint({"id": name, **result})

## Inspect authority and execution evidence

A proposal claims permission and confirmation, but its trusted app context has no permission. Another proposal claims confirmation while the app says false. Both stop before mutation. Native action success means the function returned successfully, not that its boolean policy decision was allow.

In [ ]:
pprint({"unauthorized": baseline["unauthorized"], "unconfirmed": baseline["unconfirmed"]})
assert baseline["normal"]["actual_mock_executions"] == 1
assert all(baseline[name]["actual_mock_executions"] == 0 for name in ["unauthorized", "invalid", "unconfirmed"])
assert baseline["unauthorized"]["native_action_results"][0]["return_value"] is False

## Change only the confirmation requirement

Keep the exact same proposals and app contexts. Disable require_confirmation for comparison; permission, type/range, and identity rules remain active. This removes a confirmation boundary, so it is a policy trade-off rather than a universal improvement.

In [ ]:
candidate = {}
for item in fixtures:
    action_log, action_events = [], []
    rails = build_rails(item["proposal"], item["application_context"], action_log, action_events, require_confirmation=False)
    native_events = await submit_fixture(rails)
    candidate[item["id"]] = action_outcome(action_log, action_events, native_events)

## Compare final exposed messages and actual mutations

The formerly unconfirmed authorized proposal now executes. Unauthorized and malformed proposals remain rejected. Regex context masking does not establish truth or full PII protection; these canonical-intent mock flows do not implement a complete HR application.

In [ ]:
for item in fixtures:
    name = item["id"]
    pprint({"id": name, "baseline": baseline[name]["decision"], "candidate": candidate[name]["decision"], "reason": candidate[name]["reason"], "candidate_exposed": candidate[name]["exposed"], "actual_candidate_action_log": candidate[name]["actual_action_log"]})
assert candidate["unconfirmed"]["actual_mock_executions"] == 1
assert candidate["unauthorized"]["actual_mock_executions"] == 0
assert candidate["invalid"]["actual_mock_executions"] == 0
pprint({"baseline_actual_mock_executions": sum(result["actual_mock_executions"] for result in baseline.values()), "candidate_actual_mock_executions": sum(result["actual_mock_executions"] for result in candidate.values()), "model_api_calls": 0})